# Promote the z-score partitions to `paper_z_score.parquet`

`paper_z_score.ipynb` runs one year range per job and suffixes every output, so a partial run
cannot take the bare name by accident. This is the deliberate promotion step, done as a producer
so the bare file has a reproduction path. Every `paper_z_score_<a>_<b>.parquet` partition in
`Dimensions/output/` is merged (there are no earlier vintages to exclude here); the checks are

1. the partitions are **disjoint** — no `paper_id` in two of them;
2. the merged row count equals the sum of the parts;
3. the year coverage has **no gap** between the first and last partition.

The coverage is written to `paper_z_score_provenance.json` beside the parquet.

In [ ]:
import os, sys, glob, json, time
import numpy as np, pandas as pd, duckdb
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Dimensions')
import dim_common as dim
OUT = dim.OUT

def partitions(stem):
    fs = glob.glob(f'{OUT}/{stem}_[0-9]*.parquet')
    return sorted(fs, key=lambda f: int(os.path.basename(f).rsplit('_', 2)[-2]))

ZS = partitions('paper_z_score')
ZP = partitions('z_score_pair')
assert ZS, f'no paper_z_score_<a>_<b>.parquet partitions in {OUT}'
YEARS = sorted({(int(os.path.basename(f).rsplit('_',2)[-2]),
                 int(os.path.basename(f).rsplit('_',2)[-1][:4])) for f in ZS})
COVER = (min(a for a, _ in YEARS), max(b for _, b in YEARS))
print(f'paper_z_score partitions : {len(ZS)}')
for f in ZS:
    print(f'   {os.path.basename(f):<40} {os.path.getsize(f)/1e6:>7.1f} MB')
print(f'z_score_pair partitions  : {len(ZP)}')
print(f'coverage                 : {COVER[0]}-{COVER[1]}')
gaps = [y for y in range(COVER[0], COVER[1] + 1) if not any(a <= y <= b for a, b in YEARS)]
print(f'year gaps                : {gaps if gaps else "none"}')
assert not gaps, f'the partitions do not tile {COVER[0]}-{COVER[1]}: missing {gaps}'

con = duckdb.connect()
con.execute("SET memory_limit='150GB'")
con.execute(f"SET temp_directory='{dim.CACHE}/duckdb_tmp'")
con.execute('SET preserve_insertion_order=false')

In [ ]:
%%time
dup = con.execute(f"SELECT count(*) - count(DISTINCT paper_id) FROM read_parquet({ZS!r})").fetchone()[0]
print(f'duplicate paper_id across partitions: {dup:,}')
assert dup == 0, 'partitions overlap -- merging would double-count papers'
part_rows = sum(con.execute(f"SELECT count(*) FROM read_parquet('{f}')").fetchone()[0] for f in ZS)
print(f'sum of partition rows: {part_rows:,}')

In [ ]:
%%time
t0 = time.time()
for stem, files in (('paper_z_score', ZS), ('z_score_pair', ZP)):
    if not files:
        continue
    dst = f'{OUT}/{stem}.parquet'
    order = 'paper_id' if stem == 'paper_z_score' else 'year, code_1, code_2'
    con.execute(f"""COPY (SELECT * FROM read_parquet({files!r}) ORDER BY {order})
                    TO '{dst}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)""")
    n = con.execute(f"SELECT count(*) FROM read_parquet('{dst}')").fetchone()[0]
    print(f'WROTE {dst}  ({n:,} rows, {os.path.getsize(dst)/1e9:.2f} GB)')
    if stem == 'paper_z_score':
        assert n == part_rows, f'merged {n:,} != sum of parts {part_rows:,}'
print(f'merged in {time.time()-t0:.0f}s')
prov = {'built': time.strftime('%Y-%m-%dT%H:%M:%S'), 'coverage_years': list(COVER),
        'source': 'Dimensions June 2025', 'partitions': [os.path.basename(f) for f in ZS],
        'rows': {'paper_z_score': part_rows}}
with open(f'{OUT}/paper_z_score_provenance.json', 'w') as fh:
    json.dump(prov, fh, indent=2)
print(f'WROTE {OUT}/paper_z_score_provenance.json')

In [ ]:
display(con.execute(f"""SELECT count(*) papers, round(avg(Z_median),2) mean_Zmed, round(median(Z_median),2) med_Zmed,
    round(avg(Z_10pct),2) mean_Z10, round(avg(Z_min),1) mean_Zmin, round(avg(n_pairs),1) mean_np, max(n_pairs) max_np
  FROM read_parquet('{OUT}/paper_z_score.parquet')""").fetchdf())
if ZP:
    display(con.execute(f"""SELECT p.year, count(*) AS pairs, round(avg(p.Z_score),2) AS mean_Z
      FROM read_parquet('{OUT}/z_score_pair.parquet') p GROUP BY 1 ORDER BY 1""").fetchdf().head(60))
con.close()